# Supplementary Figure: Gradient Attribution vs Experimental Effect

## Purpose
Generate publication-quality scatter plot showing correlation between
gradient-derived gene importance (mean |∂output/∂input|) and experimental
CRISPRi effect sizes (mean |log₂FC|) for GFI1B perturbation.

**Paper claim**: Pearson r = 0.83 (CDT_InSilico_KD_Improved.ipynb, Cell 18)

## Method
1. Compute Jacobian: ∂(output_j)/∂(input_i) for top 100 experimentally affected genes
2. Per input gene: mean |gradient| across 100 outputs = gradient importance score
3. Per input gene: mean |log₂FC| from experimental CRISPRi = experimental effect
4. Scatter plot: gradient importance vs experimental effect (2,361 genes)

## CRISPRi Design Application
Genes with high gradient importance but low/unknown experimental effect are
candidates for new CRISPRi validation experiments.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install h5py tqdm scipy pandas -q

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from dataclasses import dataclass
from typing import Optional
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings

# Use DejaVu Sans (available on Colab) instead of Arial
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['font.size'] = 10

# Suppress font warnings
warnings.filterwarnings('ignore', message='.*findfont.*')

print("Imports successful!")

In [ ]:
# Paths - Stage 1.5
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
MODEL_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_crispri_stage1_5")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/gradient_validation")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

TSS_EFFECTS_PATH = DRIVE_BASE / "morris_celllevel_effects_2361.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
MODEL_PATH = MODEL_BASE / "cdt_morris_celllevel_best.pt"

print("Checking files...")
for name, path in [
    ("Cell-level effects (2361)", TSS_EFFECTS_PATH),
    ("Enformer embeddings", TSS_ENFORMER_PATH),
    ("Model (Stage 1.5)", MODEL_PATH),
]:
    status = "OK" if path.exists() else "NOT FOUND"
    print(f"  [{status}] {name}")

## 2. Load Data

In [ ]:
# Load cell-level effects
with h5py.File(TSS_EFFECTS_PATH, 'r') as f:
    tss_log2fc = f['log2fc'][:]
    tss_cell_expr = f['cell_expr'][:]
    tss_target_gene_idx = f['target_gene_idx'][:]
    tss_target_gene_names = [g.decode() if isinstance(g, bytes) else g
                             for g in f['target_gene_names'][:]]
    tss_val_genes = [g.decode() if isinstance(g, bytes) else g
                     for g in f['val_genes'][:]]
    if 'ntc_mean_expr' in f.keys():
        ntc_mean_expr = f['ntc_mean_expr'][:]
    else:
        ntc_mean_expr = None

N_GENES = tss_cell_expr.shape[1]
print(f"CDT genes: {N_GENES}")
print(f"Cells: {tss_log2fc.shape[0]}")
print(f"Val genes: {tss_val_genes}")

# Load Enformer embeddings
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    tss_enformer_emb = f['embeddings'][:]
    tss_enformer_genes = [g.decode() if isinstance(g, bytes) else g
                          for g in f['gene_names'][:]]
tss_gene_to_enformer = {gene: i for i, gene in enumerate(tss_enformer_genes)}
print(f"Enformer genes: {len(tss_enformer_genes)}")

# Load gene names
GENE_LIST_PATH = DRIVE_BASE / "k562_gene_embeddings_aligned.h5"
with h5py.File(GENE_LIST_PATH, 'r') as f:
    base_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
if 'GFI1B' not in base_genes:
    cdt_genes = base_genes + ['GFI1B']
else:
    cdt_genes = base_genes
gene_to_idx = {g: i for i, g in enumerate(cdt_genes)}
print(f"Gene list: {len(cdt_genes)} genes")

# NTC mean
if ntc_mean_expr is None:
    ntc_mean_expr = tss_cell_expr.mean(axis=0)
    print("Computed NTC mean from cell expr")

## 3. Model Definition & Loading

In [ ]:
@dataclass
class CDTCRISPRiConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_genes, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim
        self.gene_embedding = nn.Embedding(n_genes, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout)
        )
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout)
        )

    def forward(self, expression):
        batch_size = expression.size(0)
        device = expression.device
        gene_ids = torch.arange(self.n_genes, device=device)
        gene_emb = self.gene_embedding(gene_ids).unsqueeze(0).expand(batch_size, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        combined = torch.cat([gene_emb, expr_emb], dim=-1)
        return self.combine(combined)


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        batch_size, query_len, _ = query.shape
        key_len = key_value.shape[1]
        Q = self.q_proj(query).view(batch_size, query_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, query_len, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model)
        )

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        batch_size = key_value.size(0)
        seq_len = key_value.size(1)
        query = query.expand(batch_size, -1, -1)
        Q = q_proj(query).view(batch_size, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj
        )
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj
        )
        concat = torch.cat([dna_pooled, rna_pooled], dim=-1)
        return self.fusion(concat)


class CDTCRISPRiModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDTCRISPRiConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)
        ])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)
        ])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_genes)
        )

    def forward(self, dna_emb, rna_expr):
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        for layer in self.dna_self_attn_layers:
            dna = layer(dna)
        for layer in self.rna_self_attn_layers:
            rna = layer(rna)
        rna = self.dna_to_rna(query=rna, key_value=dna)
        cell_embedding = self.vce(dna, rna)
        effect = self.task_layer(cell_embedding)
        return effect

print("Model defined!")

In [ ]:
# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config = CDTCRISPRiConfig()
model = CDTCRISPRiModel(config).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()
print("Model loaded!")

## 4. Compute Jacobian (Gradient Attribution)

For GFI1B perturbation: compute ∂(output_j)/∂(input_i) for top 100
experimentally affected output genes, using NTC baseline expression.

In [ ]:
# GFI1B setup
gene = 'GFI1B'
gfi1b_idx = gene_to_idx.get(gene, N_GENES - 1)
gfi1b_gene_idx_in_target = tss_target_gene_names.index('GFI1B')
gfi1b_cell_mask = tss_target_gene_idx == gfi1b_gene_idx_in_target

# Experimental effect: mean log2FC across GFI1B-perturbed cells
gfi1b_mean_log2fc = tss_log2fc[gfi1b_cell_mask].mean(axis=0)  # [2361]
experimental_effect_abs = np.abs(gfi1b_mean_log2fc)  # [2361]

# Top 100 experimentally affected genes
top_affected_indices = np.argsort(experimental_effect_abs)[-100:][::-1]

print(f"GFI1B index: {gfi1b_idx}")
print(f"GFI1B cells: {gfi1b_cell_mask.sum()}")
print(f"Top 5 affected genes: {[cdt_genes[i] for i in top_affected_indices[:5]]}")

In [ ]:
# Compute Jacobian for top 100 affected genes using NTC baseline
print("Computing Jacobian for top 100 affected genes...")

gfi1b_dna = tss_enformer_emb[tss_gene_to_enformer['GFI1B']]
dna_t = torch.from_numpy(gfi1b_dna.astype(np.float32)).unsqueeze(0).to(device)
rna_t = torch.from_numpy(ntc_mean_expr.astype(np.float32)).unsqueeze(0).to(device)
rna_t.requires_grad_(True)

jacobian_ntc = np.zeros((100, N_GENES), dtype=np.float32)

for i, out_idx in enumerate(tqdm(top_affected_indices, desc="Jacobian")):
    model.zero_grad()
    rna_t.grad = None
    pred = model(dna_t, rna_t)
    pred[0, out_idx].backward(retain_graph=True)
    jacobian_ntc[i] = rna_t.grad[0].cpu().numpy()

print(f"Jacobian shape: {jacobian_ntc.shape}")

In [ ]:
# Gradient importance: mean |gradient| per input gene across 100 outputs
mean_abs_grad = np.abs(jacobian_ntc).mean(axis=0)  # [2361]

# Reproduce r = 0.83
r_grad_exp, p_grad_exp = pearsonr(mean_abs_grad, experimental_effect_abs)
rho_grad_exp, p_rho = spearmanr(mean_abs_grad, experimental_effect_abs)

print(f"Mean |gradient| vs |experimental effect|:")
print(f"  Pearson  r = {r_grad_exp:.4f} (p = {p_grad_exp:.2e})")
print(f"  Spearman ρ = {rho_grad_exp:.4f} (p = {p_rho:.2e})")

## 5. Publication-Quality Scatter Plot

Supplementary Figure: Gradient attribution validates experimental regulatory relationships

In [ ]:
# ── Build DataFrame for plotting ──
df = pd.DataFrame({
    'gene': cdt_genes,
    'gradient_importance': mean_abs_grad,
    'experimental_effect': experimental_effect_abs,
    'signed_effect': gfi1b_mean_log2fc,
})

# Mark categories
df['category'] = 'Other'
df.loc[df['gene'] == 'GFI1B', 'category'] = 'GFI1B (target)'
for vg in tss_val_genes:
    if vg != 'GFI1B':
        df.loc[df['gene'] == vg, 'category'] = 'Validation gene'

# Top gradient genes (top 20) that are NOT in val set
top_grad_idx = np.argsort(mean_abs_grad)[-20:][::-1]
top_grad_genes = [cdt_genes[i] for i in top_grad_idx]
for g in top_grad_genes:
    if df.loc[df['gene'] == g, 'category'].values[0] == 'Other':
        df.loc[df['gene'] == g, 'category'] = 'High gradient'

print(f"Categories: {df['category'].value_counts().to_dict()}")
print(f"\nTop 10 by gradient importance:")
print(df.nlargest(10, 'gradient_importance')[['gene', 'gradient_importance', 'experimental_effect']].to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MAIN FIGURE: Gradient Attribution vs Experimental Effect
# ══════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(6, 5))

# Plot "Other" genes first (background)
mask_other = df['category'] == 'Other'
ax.scatter(df.loc[mask_other, 'experimental_effect'],
           df.loc[mask_other, 'gradient_importance'],
           s=8, alpha=0.3, c='#999999', label=f'Other genes (n={mask_other.sum()})',
           edgecolors='none', rasterized=True)

# High gradient genes (no individual labels — legend is enough)
mask_high = df['category'] == 'High gradient'
ax.scatter(df.loc[mask_high, 'experimental_effect'],
           df.loc[mask_high, 'gradient_importance'],
           s=40, alpha=0.8, c='#E67E22', label='High gradient importance',
           edgecolors='black', linewidths=0.5, zorder=3)

# Validation genes
mask_val = df['category'] == 'Validation gene'
ax.scatter(df.loc[mask_val, 'experimental_effect'],
           df.loc[mask_val, 'gradient_importance'],
           s=60, alpha=0.9, c='#2980B9', marker='D', label='Held-out validation genes',
           edgecolors='black', linewidths=0.5, zorder=4)

# GFI1B
mask_gfi1b = df['category'] == 'GFI1B (target)'
ax.scatter(df.loc[mask_gfi1b, 'experimental_effect'],
           df.loc[mask_gfi1b, 'gradient_importance'],
           s=100, alpha=1.0, c='#E74C3C', marker='*', label='GFI1B (perturbation target)',
           edgecolors='black', linewidths=0.5, zorder=5)

# Regression line
x_range = np.linspace(0, df['experimental_effect'].max() * 1.05, 100)
slope, intercept = np.polyfit(df['experimental_effect'], df['gradient_importance'], 1)
ax.plot(x_range, slope * x_range + intercept, '--', color='#E74C3C', alpha=0.6, linewidth=1.5)

# Annotate ONLY validation genes (GFI1B + 4 others) — clean and minimal
annotate_genes = ['GFI1B'] + [g for g in tss_val_genes if g != 'GFI1B']

for g in annotate_genes:
    row = df.loc[df['gene'] == g]
    if len(row) == 0:
        continue
    x, y = row['experimental_effect'].values[0], row['gradient_importance'].values[0]
    # Place label to the right of the point
    ax.annotate(g, (x, y), fontsize=7, fontweight='bold',
                xytext=(8, 0), textcoords='offset points',
                va='center',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor='gray', alpha=0.8))

# Stats annotation
ax.text(0.05, 0.95, f'Pearson r = {r_grad_exp:.2f}\n(n = {len(df):,} genes)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.set_xlabel('Experimental effect size (mean |log$_2$FC|)', fontsize=11)
ax.set_ylabel('Gradient importance (mean |$\partial$output/$\partial$input|)', fontsize=11)
ax.set_title('Gradient Attribution Validates Experimental\nRegulatory Relationships (GFI1B)', fontsize=12)
ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'gradient_vs_experimental_scatter.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_BASE / 'gradient_vs_experimental_scatter.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_BASE}")

## 6. Ranked Gene List for CRISPRi Experiment Design

Genes ranked by gradient importance = predicted regulatory influence.
High-gradient genes with low/absent experimental validation are
priority candidates for new CRISPRi experiments.

**Rationale**: If the gradient says gene X strongly influences downstream
expression (high ∂output/∂input), but we haven't perturbed gene X
experimentally, it's a candidate for the next CRISPRi screen.

In [ ]:
# ── Full ranked list ──
df_ranked = df[['gene', 'gradient_importance', 'experimental_effect', 'signed_effect']].copy()
df_ranked['gradient_rank'] = df_ranked['gradient_importance'].rank(ascending=False).astype(int)
df_ranked['experimental_rank'] = df_ranked['experimental_effect'].rank(ascending=False).astype(int)
df_ranked['rank_difference'] = df_ranked['gradient_rank'] - df_ranked['experimental_rank']

# Genes where gradient rank >> experimental rank = model predicts high influence
# but experiment shows low effect → potential novel regulatory targets
df_ranked['discovery_score'] = (
    df_ranked['gradient_importance'] / (df_ranked['gradient_importance'].max() + 1e-10)
    - df_ranked['experimental_effect'] / (df_ranked['experimental_effect'].max() + 1e-10)
)

df_ranked = df_ranked.sort_values('gradient_rank')

print("Top 30 genes by gradient importance:")
print("="*90)
print(f"{'Rank':>4} {'Gene':<12} {'|Gradient|':>10} {'|Exp Effect|':>12} {'Grad Rank':>10} {'Exp Rank':>10}")
print("-"*90)
for _, row in df_ranked.head(30).iterrows():
    marker = ' ***' if row['gradient_rank'] <= 30 and row['experimental_rank'] > 100 else ''
    print(f"{row['gradient_rank']:4.0f} {row['gene']:<12} {row['gradient_importance']:10.6f} "
          f"{row['experimental_effect']:12.6f} {row['gradient_rank']:10.0f} {row['experimental_rank']:10.0f}{marker}")

print("\n*** = High gradient, low experimental rank → CRISPRi validation candidate")

In [ ]:
# ── CRISPRi candidates: high gradient, low experimental effect ──
# These are genes the model predicts as regulatory hubs but haven't been
# strongly perturbed in the Morris dataset

crispri_candidates = df_ranked[
    (df_ranked['gradient_rank'] <= 50) &
    (df_ranked['experimental_rank'] > 200)
].sort_values('gradient_rank')

print(f"\nCRISPRi Validation Candidates")
print(f"(Top 50 by gradient importance, BUT ranked >200 by experimental effect)")
print("="*70)
if len(crispri_candidates) > 0:
    for _, row in crispri_candidates.iterrows():
        print(f"  {row['gene']:<12} gradient_rank={row['gradient_rank']:.0f}, "
              f"exp_rank={row['experimental_rank']:.0f}, "
              f"|grad|={row['gradient_importance']:.6f}")
else:
    print("  No candidates found (gradient and experimental rankings are well-aligned)")
    # In that case, show genes with highest discovery_score
    print("\n  Top 10 by discovery score (gradient outranks experimental):")
    for _, row in df_ranked.nlargest(10, 'discovery_score').iterrows():
        print(f"  {row['gene']:<12} grad_rank={row['gradient_rank']:.0f}, "
              f"exp_rank={row['experimental_rank']:.0f}, "
              f"discovery_score={row['discovery_score']:.4f}")

In [ ]:
# ── Save full ranked list as CSV ──
csv_path = OUTPUT_BASE / 'gradient_ranked_genes_gfi1b.csv'
df_ranked.to_csv(csv_path, index=False)
print(f"Saved ranked gene list: {csv_path}")

# ── Save Jacobian for future analysis ──
npz_path = OUTPUT_BASE / 'jacobian_gfi1b_top100.npz'
np.savez_compressed(npz_path,
    jacobian=jacobian_ntc,
    output_gene_indices=top_affected_indices,
    output_gene_names=np.array([cdt_genes[i] for i in top_affected_indices]),
    input_gene_names=np.array(cdt_genes),
    mean_abs_gradient=mean_abs_grad,
    experimental_effect_abs=experimental_effect_abs,
    pearson_r=r_grad_exp,
    spearman_rho=rho_grad_exp,
)
print(f"Saved Jacobian data: {npz_path}")

## 7. Extended Analysis: Per-Validation-Gene Gradient

Repeat gradient analysis for each validation gene to show generalizability.

In [ ]:
# ── Gradient analysis for all 5 validation genes ──
# Debug: check which val genes are available
print("Checking validation gene availability:")
for vg in tss_val_genes:
    in_enformer = vg in tss_gene_to_enformer
    in_gene_list = vg in gene_to_idx
    in_targets = vg in tss_target_gene_names
    print(f"  {vg}: Enformer={in_enformer}, gene_list={in_gene_list}, targets={in_targets}")

val_results = []

for val_gene in tss_val_genes:
    if val_gene not in tss_gene_to_enformer:
        print(f"  {val_gene}: not in Enformer embeddings, skipping")
        continue

    if val_gene not in tss_target_gene_names:
        print(f"  {val_gene}: not in target gene names, skipping")
        continue

    # Get cells and experimental effects for this gene
    val_gene_idx_in_target = tss_target_gene_names.index(val_gene)
    val_cell_mask = tss_target_gene_idx == val_gene_idx_in_target
    n_cells = int(val_cell_mask.sum())
    if n_cells == 0:
        print(f"  {val_gene}: 0 cells in dataset, skipping")
        continue

    val_mean_log2fc = tss_log2fc[val_cell_mask].mean(axis=0)
    val_exp_abs = np.abs(val_mean_log2fc)
    
    # Top 100 affected genes for this perturbation
    val_top_affected = np.argsort(val_exp_abs)[-100:][::-1]
    
    # Compute Jacobian
    val_dna = tss_enformer_emb[tss_gene_to_enformer[val_gene]]
    dna_t = torch.from_numpy(val_dna.astype(np.float32)).unsqueeze(0).to(device)
    rna_t = torch.from_numpy(ntc_mean_expr.astype(np.float32)).unsqueeze(0).to(device)
    rna_t.requires_grad_(True)
    
    val_jacobian = np.zeros((100, N_GENES), dtype=np.float32)
    for i, out_idx in enumerate(tqdm(val_top_affected, desc=f"{val_gene} Jacobian")):
        model.zero_grad()
        rna_t.grad = None
        pred = model(dna_t, rna_t)
        pred[0, out_idx].backward(retain_graph=True)
        val_jacobian[i] = rna_t.grad[0].cpu().numpy()
    
    val_mean_grad = np.abs(val_jacobian).mean(axis=0)
    r, p = pearsonr(val_mean_grad, val_exp_abs)
    rho, p_rho = spearmanr(val_mean_grad, val_exp_abs)
    
    val_results.append({
        'gene': val_gene,
        'n_cells': n_cells,
        'pearson_r': r,
        'pearson_p': p,
        'spearman_rho': rho,
    })
    print(f"  {val_gene} ({n_cells} cells): Pearson r = {r:.4f}, Spearman ρ = {rho:.4f}")

df_val = pd.DataFrame(val_results)
print(f"\nProcessed {len(df_val)}/{len(tss_val_genes)} validation genes")
print(f"Mean Pearson r: {df_val['pearson_r'].mean():.4f}")
print(df_val.to_string(index=False))

In [ ]:
# ── Bar chart: Per-gene gradient correlation ──
if len(df_val) > 0:
    fig, ax = plt.subplots(figsize=(5, 3.5))
    
    colors = ['#E74C3C' if g == 'GFI1B' else '#2980B9' for g in df_val['gene']]
    bars = ax.bar(df_val['gene'], df_val['pearson_r'], color=colors, edgecolor='black', linewidth=0.5)
    
    mean_r = df_val['pearson_r'].mean()
    ax.axhline(y=mean_r, color='gray', linestyle='--', linewidth=1)
    
    ax.set_ylabel('Pearson r\n(gradient vs experimental)', fontsize=10)
    ax.set_xlabel('Perturbation target gene', fontsize=10)
    ax.set_title(f'Gradient Attribution Accuracy Across Validation Genes\n(mean r = {mean_r:.2f})',
                 fontsize=11)
    ax.set_ylim(0, 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add r values on bars
    for bar, r_val in zip(bars, df_val['pearson_r']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{r_val:.2f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_BASE / 'gradient_correlation_per_gene.pdf', bbox_inches='tight')
    plt.savefig(OUTPUT_BASE / 'gradient_correlation_per_gene.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved per-gene bar chart")

## 8. Jacobian Heatmap: Gene-Gene Regulatory Map

Visualize the Jacobian matrix as a clustered heatmap to reveal which input genes
influence which output genes under GFI1B perturbation.

- **Rows**: Top 100 experimentally affected output genes (∂output_j)
- **Columns**: Top 50 input genes by gradient importance (∂/∂input_i)
- **Values**: |∂output_j / ∂input_i| — regulatory influence strength
- **Clustering**: Hierarchical clustering on both axes to reveal regulatory modules

In [ ]:
# ══════════════════════════════════════════════════════════════════
# JACOBIAN HEATMAP: Gene-Gene Regulatory Map
# ══════════════════════════════════════════════════════════════════
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import pdist
from matplotlib.colors import LinearSegmentedColormap

# --- Select top input genes by gradient importance ---
N_INPUT_GENES = 50  # columns
abs_jacobian = np.abs(jacobian_ntc)  # [100 output × 2361 input]

# Top 50 input genes by mean |gradient| across 100 outputs
top_input_indices = np.argsort(mean_abs_grad)[-N_INPUT_GENES:][::-1]
top_input_names = [cdt_genes[i] for i in top_input_indices]

# Output gene names (top 100 experimentally affected)
top_output_names = [cdt_genes[i] for i in top_affected_indices]

# Submatrix: 100 outputs × 50 inputs
heatmap_data = abs_jacobian[:, top_input_indices]

print(f"Heatmap matrix: {heatmap_data.shape}")
print(f"Output genes (rows): {len(top_output_names)}")
print(f"Input genes (cols): {len(top_input_names)}")

# --- Log-transform for better visualization ---
heatmap_log = np.log10(heatmap_data + 1e-8)

# --- Hierarchical clustering ---
row_linkage = linkage(pdist(heatmap_log, metric='correlation'), method='ward')
row_order = leaves_list(row_linkage)

col_linkage = linkage(pdist(heatmap_log.T, metric='correlation'), method='ward')
col_order = leaves_list(col_linkage)

# Reorder matrix
heatmap_clustered = heatmap_log[row_order][:, col_order]
row_labels = [top_output_names[i] for i in row_order]
col_labels = [top_input_names[i] for i in col_order]

# --- Mark special genes ---
val_gene_set = set(tss_val_genes)
target_gene_set = set(tss_target_gene_names)

# --- Custom colormap: blue (no influence) → white → red (strong influence) ---
cmap_blue_red = LinearSegmentedColormap.from_list(
    'blue_white_red',
    ['#2166AC', '#67A9CF', '#D1E5F0', '#FDDBC7', '#EF8A62', '#B2182B'],
    N=256
)

# --- Plot ---
fig, ax = plt.subplots(figsize=(16, 20))

im = ax.imshow(heatmap_clustered, aspect='auto', cmap=cmap_blue_red,
               interpolation='nearest')

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label('log$_{10}$|$\\partial$output/$\\partial$input|', fontsize=11)

# Y-axis: output genes (rows)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=5.5)

# X-axis: input genes (columns)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=7, rotation=90)

# Highlight validation genes in labels
for i, label in enumerate(col_labels):
    if label in val_gene_set:
        ax.get_xticklabels()[i].set_color('#E74C3C')
        ax.get_xticklabels()[i].set_fontweight('bold')
    elif label in target_gene_set:
        ax.get_xticklabels()[i].set_color('#2980B9')
        ax.get_xticklabels()[i].set_fontweight('bold')

for i, label in enumerate(row_labels):
    if label in val_gene_set:
        ax.get_yticklabels()[i].set_color('#E74C3C')
        ax.get_yticklabels()[i].set_fontweight('bold')
    elif label in target_gene_set:
        ax.get_yticklabels()[i].set_color('#2980B9')
        ax.get_yticklabels()[i].set_fontweight('bold')

ax.set_xlabel('Input genes (top 50 by gradient importance)', fontsize=12)
ax.set_ylabel('Output genes (top 100 experimentally affected)', fontsize=12)
ax.set_title('Jacobian Regulatory Map: GFI1B Perturbation\n'
             '|$\\partial$(output gene) / $\\partial$(input gene)| at NTC baseline',
             fontsize=14)

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'fig8c_jacobian_heatmap.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_BASE / 'fig8c_jacobian_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSaved: fig8c_jacobian_heatmap.pdf/png")
print(f"Blue = no regulatory influence, Red = strong regulatory influence")

## 9. Summary

In [ ]:
print("="*70)
print("GRADIENT ATTRIBUTION VALIDATION - SUMMARY")
print("="*70)

print(f"\nGFI1B perturbation:")
print(f"  Genes: {N_GENES}")
print(f"  Jacobian: top 100 affected output genes × {N_GENES} input genes")
print(f"  Mean |gradient| vs |experimental effect|:")
print(f"    Pearson  r = {r_grad_exp:.4f}")
print(f"    Spearman ρ = {rho_grad_exp:.4f}")

if len(df_val) > 0:
    print(f"\nPer-validation-gene results:")
    for _, row in df_val.iterrows():
        print(f"  {row['gene']:<10} r = {row['pearson_r']:.4f}")
    print(f"  {'Mean':<10} r = {df_val['pearson_r'].mean():.4f}")

print(f"\nOutputs saved to: {OUTPUT_BASE}")
print(f"  - gradient_vs_experimental_scatter.pdf/png  (Fig 8A)")
print(f"  - gradient_correlation_per_gene.pdf/png     (Fig 8B)")
print(f"  - fig8c_jacobian_heatmap.pdf/png             (Fig 8C)")
print(f"  - gradient_ranked_genes_gfi1b.csv")
print(f"  - jacobian_gfi1b_top100.npz")
print("\n" + "="*70)